In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import os

# =====================
# CONFIG
# =====================
DATA_DIR = "eyes/eyes"
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", DEVICE)

# =====================
# TRANSFORMS
# =====================
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

# =====================
# DATASETS
# =====================
train_data = datasets.ImageFolder(os.path.join(DATA_DIR, "train"), transform=transform)
val_data   = datasets.ImageFolder(os.path.join(DATA_DIR, "val"), transform=transform)
test_data  = datasets.ImageFolder(os.path.join(DATA_DIR, "test"), transform=transform)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_data, batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_data, batch_size=BATCH_SIZE)

class_names = train_data.classes
print("Classes:", class_names)

# =====================
# MODEL (MobileNetV2)
# =====================
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)

# Modify classifier
model.classifier[1] = nn.Linear(model.last_channel, 2)

model = model.to(DEVICE)

# =====================
# LOSS & OPTIMIZER
# =====================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# =====================
# TRAIN LOOP
# =====================
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    correct = 0

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()

    acc = correct / len(train_data)

    # VALIDATION
    model.eval()
    val_correct = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            val_correct += (preds == labels).sum().item()

    val_acc = val_correct / len(val_data)

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {total_loss:.4f} | Train Acc: {acc:.4f} | Val Acc: {val_acc:.4f}")

# =====================
# SAVE MODEL
# =====================
torch.save(model.state_dict(), "drowsy_mobilenetv2.pth")
print("Model saved!")

Using device: cuda
Classes: ['Close', 'Open']
Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to C:\Users\Ali Mohamed/.cache\torch\hub\checkpoints\mobilenet_v2-7ebf99e0.pth


100.0%


Epoch 1/10 | Loss: 123.6167 | Train Acc: 0.9732 | Val Acc: 0.9738
Epoch 2/10 | Loss: 82.6277 | Train Acc: 0.9830 | Val Acc: 0.9846
Epoch 3/10 | Loss: 76.9231 | Train Acc: 0.9840 | Val Acc: 0.9797
Epoch 4/10 | Loss: 72.6028 | Train Acc: 0.9846 | Val Acc: 0.9626
Epoch 5/10 | Loss: 63.4222 | Train Acc: 0.9863 | Val Acc: 0.9876
Epoch 6/10 | Loss: 61.3038 | Train Acc: 0.9867 | Val Acc: 0.9830
Epoch 7/10 | Loss: 60.1246 | Train Acc: 0.9868 | Val Acc: 0.9866
Epoch 8/10 | Loss: 54.2782 | Train Acc: 0.9882 | Val Acc: 0.9875
Epoch 9/10 | Loss: 53.3223 | Train Acc: 0.9885 | Val Acc: 0.9877
Epoch 10/10 | Loss: 49.5697 | Train Acc: 0.9891 | Val Acc: 0.9852
Model saved!
